# Hallmark Pathway Activity in Macrophage States (AUCell)

Scores MSigDB Hallmark gene sets per cell with AUCell (omicverse) in the three major
macrophage states (INHBAhigh, CD36high, STAB1high), excluding liver biopsies. Pathway
activity is compared between states, and between state × treatment-stage groups, with
Wilcoxon tests on AUC scores; a per-patient summary is added as a robustness check.

In [ ]:
# !pip install --quiet scanpy leidenalg omicverse
# %cd /content/drive/MyDrive/Colab Notebooks/data/h5ad/myeloid/

In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
import omicverse as ov
import pandas as pd
import scanpy as sc
import seaborn as sns
from scipy.cluster.hierarchy import linkage

## Config

In [ ]:
input_path = "sub_Myeloid_raw_scvi_combine_annotated.h5ad"

geneset_path = "genesets/h.all.v2025.1.Hs.symbols.gmt"        # MSigDB Hallmark v2025.1
figures_output_dir = "./figures"
tables_output_dir = "./tables"

counts_layer = "raw_counts"
state_col = "cell_state"
recist_col = "recist_v3"
donor_col = "ID_2"
liver_biopsy_ids = ["R008", "R011"]                          # matched on obs["ID"]; excluded

macrophage_states = ["Macrophage INHBAhigh", "Macrophage CD36high", "Macrophage STAB1high"] #Here we focus on the main population only
recist_order = ["0_adj_normal", "1_treatment_naive", "3_on_treatment", "4_TKI_resistance"]
aucell_workers = 8
seed = 0

# --- state heatmap (Wilcoxon z-score of each state vs the other states) ---
state_padj_filter = 1e-3            # pathway-state pairs kept for the heatmap (others shown as 0)
state_star = {"padj": 1e-3, "score": 1}
# pathways removed from the state heatmap (curated; report in Methods)
excluded_pathways = [
    "HALLMARK_APICAL_JUNCTION", "HALLMARK_MYC_TARGETS_V1", "HALLMARK_DNA_REPAIR",
    "HALLMARK_UNFOLDED_PROTEIN_RESPONSE", "HALLMARK_PROTEIN_SECRETION", "HALLMARK_MTORC1_SIGNALING",
    "HALLMARK_PI3K_AKT_MTOR_SIGNALING", "HALLMARK_ANDROGEN_RESPONSE", "HALLMARK_ESTROGEN_RESPONSE_EARLY",
    "HALLMARK_ESTROGEN_RESPONSE_LATE", "HALLMARK_UV_RESPONSE_DN", "HALLMARK_UV_RESPONSE_UP",
    "HALLMARK_SPERMATOGENESIS", "HALLMARK_KRAS_SIGNALING_UP", "HALLMARK_KRAS_SIGNALING_DN",
]

# --- state x treatment heatmap ---
treatment_pathways = [
    "HALLMARK_BILE_ACID_METABOLISM", "HALLMARK_HEME_METABOLISM", "HALLMARK_IL2_STAT5_SIGNALING",
    "HALLMARK_INFLAMMATORY_RESPONSE", "HALLMARK_TNFA_SIGNALING_VIA_NFKB", "HALLMARK_HYPOXIA",
    "HALLMARK_APOPTOSIS", "HALLMARK_ANGIOGENESIS", "HALLMARK_IL6_JAK_STAT3_SIGNALING",
    "HALLMARK_INTERFERON_ALPHA_RESPONSE", "HALLMARK_INTERFERON_GAMMA_RESPONSE", "HALLMARK_GLYCOLYSIS",
    "HALLMARK_CHOLESTEROL_HOMEOSTASIS", "HALLMARK_REACTIVE_OXYGEN_SPECIES_PATHWAY",
    "HALLMARK_FATTY_ACID_METABOLISM", "HALLMARK_OXIDATIVE_PHOSPHORYLATION", "HALLMARK_TGF_BETA_SIGNALING",
    "HALLMARK_HEDGEHOG_SIGNALING", "HALLMARK_NOTCH_SIGNALING", "HALLMARK_WNT_BETA_CATENIN_SIGNALING",
    "HALLMARK_COMPLEMENT", "HALLMARK_ALLOGRAFT_REJECTION",
]
treatment_star = {"padj": 0.05, "score": 6}


# --- marker dotplots (all myeloid states) ---
mdsc_panel = {"MDSC": ["CD14", "ITGAM", "CD33", "CD274", "FUT4"],
              "HLA": ["HLA-DRA", "HLA-DRB1", "HLA-DQA1"]}

macrophage_signature_panel = {
    "core-2": ["CD163", "MRC1", "PPARG", "LSAMP"],
    "core-1": ["STAB1", "CD36", "LGMN"],
    "core": ["CD14", "FCGR3A", "VCAN", "LYZ", "IL1B", "TNF", "CLEC12A", "S100A8", "S100A9", "FCN1"],
    "both": ["FRMD4B", "ABCA1", "PRKAG2", "STARD13", "GAS7", "CD109", "SGK1", "SRGAP1", "PPM1L", "PMP22"],
    "cd36high": ["KCNMA1", "FMN1", "FKBP5", "TFRC", "ACSL1", "CCL18", "GPNMB", "PLPP3"],
    "stab1high": ["RBPJ", "PDGFC", "F13A1", "IL2RA", "CD163L1", "SLC40A1", "PDE4D", "ME1", "EDA", "ABCA6"],
}

os.makedirs(figures_output_dir, exist_ok=True)
os.makedirs(tables_output_dir, exist_ok=True)
sc.settings.figdir = figures_output_dir
ov.plot_set()

## 1. Load, remove liver biopsies and normalise

In [ ]:
adata = sc.read(input_path)
adata.X = adata.layers[counts_layer].copy()
adata.uns.pop("log1p", None)
adata = adata[~adata.obs["ID"].isin(liver_biopsy_ids)].copy()
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
print(adata.n_obs, "myeloid cells after removing liver biopsies")
adata.obs[state_col].value_counts()

## 2. Marker dotplots across myeloid states

In [ ]:
def marker_dotplot(ad, panel, figsize, save_path=None):
    present = {k: [g for g in dict.fromkeys(v) if g in ad.var_names] for k, v in panel.items()}
    missing = {k: sorted(set(v) - set(present[k])) for k, v in panel.items()}
    if any(missing.values()):
        print("not in data:", {k: v for k, v in missing.items() if v})
    axes_dict = sc.pl.dotplot(ad, {k: v for k, v in present.items() if v}, groupby=state_col, use_raw=False,
                              standard_scale="var", cmap="Blues", figsize=figsize, dendrogram=True, show=False)
    for ax in axes_dict.values():
        for tick in ax.get_xticklabels():
            tick.set_style("italic")
    if save_path:
        plt.savefig(save_path, bbox_inches="tight")
    plt.show()

marker_dotplot(adata, mdsc_panel, (4.5, 2.4), save_path=f"{figures_output_dir}/dotplot_selected_MDSC_lung.pdf")
marker_dotplot(adata, macrophage_signature_panel, (13, 3))

## 3. Subset to macrophage states

In [ ]:
adata = adata[adata.obs[state_col].isin(macrophage_states)].copy()
adata.obs[state_col] = pd.Categorical(adata.obs[state_col].astype(str), categories=macrophage_states)
adata.obs["cell_treatment"] = pd.Categorical(
    adata.obs[state_col].astype(str) + "-" + adata.obs[recist_col].astype(str),
    categories=[f"{s}-{r}" for s in macrophage_states for r in recist_order
                if ((adata.obs[state_col] == s) & (adata.obs[recist_col] == r)).any()])
print(adata.n_obs, "macrophages")
print(pd.crosstab(adata.obs[state_col], adata.obs[recist_col]))

fig, axes = plt.subplots(len(recist_order), 1, figsize=(3.5, 3 * len(recist_order)))
for ax, stage in zip(axes, recist_order):
    sc.pl.umap(adata[adata.obs[recist_col] == stage], color=state_col, size=20, ax=ax, show=False, title=stage)
plt.tight_layout()
plt.show()

## 4. AUCell pathway scores

Computed once on log-normalised expression and reused for all comparisons below
(the original recomputed them for each comparison with identical inputs).

In [ ]:
pathways = ov.utils.geneset_prepare(geneset_path, organism="Human")
adata_auc = ov.single.pathway_aucell_enrichment(adata, pathways_dict=pathways, num_workers=aucell_workers)
adata_auc.obs = adata.obs.loc[adata_auc.obs_names].copy()
adata_auc.obsm = adata[adata_auc.obs_names].obsm.copy()
auc = adata_auc.to_df()
auc.to_csv(f"{tables_output_dir}/AUCell_hallmark_macrophage_scores.csv.gz")
print(auc.shape[1], "pathways scored")

## 5. Pathways by macrophage state

Wilcoxon test of AUC scores for each state against the other two. `scores` are Wilcoxon
z-statistics (positive = higher in that state). Pathway–state pairs with padj ≥
`state_padj_filter` are shown as 0; `*` marks padj < `state_star['padj']` and z > `state_star['score']`.

In [ ]:
def rank_pathways(groupby, csv_name):
    sc.tl.rank_genes_groups(adata_auc, groupby, method="wilcoxon", n_genes=adata_auc.n_vars)
    df = sc.get.rank_genes_groups_df(adata_auc, group=None)
    df = df.drop(columns=[c for c in ("logfoldchanges",) if c in df])   # not meaningful for AUC scores
    df.to_csv(f"{tables_output_dir}/{csv_name}", index=False)
    return df

def star_matrix(score, padj, rule):
    mask = (padj.reindex_like(score) < rule["padj"]) & (score > rule["score"])
    return pd.DataFrame(np.where(mask, "*", ""), index=score.index, columns=score.columns)

state_rank = rank_pathways(state_col, "AUCell_pathway_differential_macrophage_cell_state_hallmark.csv")
kept = state_rank[state_rank["pvals_adj"] < state_padj_filter]
score = kept.pivot(index="names", columns="group", values="scores").fillna(0)
padj = kept.pivot(index="names", columns="group", values="pvals_adj").reindex_like(score)
score = score.drop(index=excluded_pathways, errors="ignore")
padj = padj.reindex_like(score)
score = score[[s for s in macrophage_states if s in score.columns]]
padj = padj[score.columns]
print(f"{len(score)} pathways shown; excluded by config: {sorted(set(excluded_pathways) & set(auc.columns))}")

g = sns.clustermap(score, cmap="RdYlGn_r", row_linkage=linkage(score, method="average"), col_cluster=False,
                   annot=star_matrix(score, padj, state_star), fmt="", figsize=(1.8, 13), linewidths=0.3,
                   cbar_kws={"label": "Wilcoxon z"})
g.ax_heatmap.set_yticklabels(g.ax_heatmap.get_yticklabels(), fontsize=8)
g.ax_heatmap.set(xlabel="", ylabel="Pathways")
g.fig.suptitle("Hallmark AUCell: state vs other states", fontsize=10, y=1.02)
g.savefig(f"{figures_output_dir}/pathway_AUCell_hallmark_cell_state.pdf", dpi=150, bbox_inches="tight")
plt.show() #Figure 3

## 6. Pathways by macrophage state × treatment stage

Each state × stage group is tested against all other groups, so a high score can
reflect the state as much as the stage; compare columns within the same state.
`*` marks padj < `treatment_star['padj']` and z > `treatment_star['score']`.

In [ ]:
treat_rank = rank_pathways("cell_treatment", "AUCell_pathway_differential_macrophage_cell_treatment_hallmark.csv")
sub = treat_rank[treat_rank["names"].isin(treatment_pathways)]
score = sub.pivot(index="names", columns="group", values="scores").reindex(treatment_pathways)
padj = sub.pivot(index="names", columns="group", values="pvals_adj").reindex_like(score)
columns = [c for c in adata_auc.obs["cell_treatment"].cat.categories if c in score.columns]
score, padj = score[columns], padj[columns]

g = sns.clustermap(score.fillna(0), cmap="RdYlGn_r", row_linkage=linkage(score.fillna(0), method="average"),
                   col_cluster=False, annot=star_matrix(score, padj, treatment_star), fmt="",
                   figsize=(12, 15), linewidths=0.3, cbar_kws={"label": "Wilcoxon z"})
g.ax_heatmap.set(xlabel="", ylabel="Pathways")
g.fig.suptitle("Hallmark AUCell by macrophage state + treatment stage", fontsize=12, y=1.02)
g.savefig(f"{figures_output_dir}/pathway_AUCell_hallmark_pattern_order.pdf", dpi=150, bbox_inches="tight")
plt.show()